In [1]:

import pandas as pd
from pathlib import Path

# Locate the project directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Build the CSV file path
file_path = project_root / "data" / "raw" / "telecom_customer_churn.csv"

# Load the CSV into a DataFrame
df = pd.read_csv(
    file_path,
    keep_default_na=False,
    na_values=[""]
)

# Display basic information
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

# Preview the first 5 records
display(df.head())


Total rows: 7043
Total columns: 38


,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [2]:

# Inspect the DataFrame structure
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Gender                             7043 non-null   str    
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   str    
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   str    
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer                              7043 non-null   str    
 12  Pho

In [3]:

# Calculate missing values for each column
missing_values = df.isna().sum()

# Calculate the percentage of missing values
missing_percentage = (missing_values / len(df)) * 100

# Create a data quality summary
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage.round(2)
})

# Sort columns by number of missing values
missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

# Display columns with missing values
display(missing_summary[missing_summary["Missing Values"] > 0])


,Missing Values,Missing Percentage
Churn Reason,5174,73.46
Churn Category,5174,73.46
Online Backup,1526,21.67
Online Security,1526,21.67
Avg Monthly GB Download,1526,21.67
Internet Type,1526,21.67
Streaming Movies,1526,21.67
Streaming TV,1526,21.67
Unlimited Data,1526,21.67
Streaming Music,1526,21.67


In [4]:

# Check how many customers have internet service
print("Internet Service Distribution:")
print(df["Internet Service"].value_counts(dropna=False))

# Compare Internet Service with missing Internet Type
print("\nMissing Internet Type by Internet Service:")

internet_missing = pd.crosstab(
    df["Internet Service"],
    df["Internet Type"].isna(),
    rownames=["Internet Service"],
    colnames=["Internet Type is Missing"],
    margins=True
)

display(internet_missing)


Internet Service Distribution:
Internet Service
Yes    5517
No     1526
Name: count, dtype: int64

Missing Internet Type by Internet Service:


Internet Type is Missing,False,True,All
Internet Service,,,
No,0,1526,1526
Yes,5517,0,5517
All,5517,1526,7043


In [5]:

# Internet-related columns
internet_columns = [
    "Internet Type",
    "Avg Monthly GB Download",
    "Online Security",
    "Online Backup",
    "Device Protection Plan",
    "Premium Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Streaming Music",
    "Unlimited Data"
]

# Identify customers with and without internet service
no_internet = df["Internet Service"] == "No"
has_internet = df["Internet Service"] == "Yes"

# Validate missing values for each column
results = []

for column in internet_columns:
    results.append({
        "Column": column,
        "Missing - No Internet": df.loc[
            no_internet, column
        ].isna().sum(),
        "Present - No Internet": df.loc[
            no_internet, column
        ].notna().sum(),
        "Missing - Has Internet": df.loc[
            has_internet, column
        ].isna().sum()
    })

# Display validation results
internet_validation = pd.DataFrame(results)

display(internet_validation)


,Column,Missing - No Internet,Present - No Internet,Missing - Has Internet
0,Internet Type,1526,0,0
1,Avg Monthly GB Download,1526,0,0
2,Online Security,1526,0,0
3,Online Backup,1526,0,0
4,Device Protection Plan,1526,0,0
5,Premium Tech Support,1526,0,0
6,Streaming TV,1526,0,0
7,Streaming Movies,1526,0,0
8,Streaming Music,1526,0,0
9,Unlimited Data,1526,0,0


In [6]:

# Check Phone Service distribution
print("Phone Service Distribution:")
print(df["Phone Service"].value_counts(dropna=False))

# Columns related to phone services
phone_columns = [
    "Multiple Lines",
    "Avg Monthly Long Distance Charges"
]

# Cross-tabulate missing values by Phone Service
for column in phone_columns:
    print(f"\nMissing values for: {column}")

    phone_missing = pd.crosstab(
        df["Phone Service"],
        df[column].isna(),
        rownames=["Phone Service"],
        colnames=["Is Missing"],
        margins=True
    )

    display(phone_missing)


Phone Service Distribution:
Phone Service
Yes    6361
No      682
Name: count, dtype: int64

Missing values for: Multiple Lines


Is Missing,False,True,All
Phone Service,,,
No,0,682,682
Yes,6361,0,6361
All,6361,682,7043



Missing values for: Avg Monthly Long Distance Charges


Is Missing,False,True,All
Phone Service,,,
No,0,682,682
Yes,6361,0,6361
All,6361,682,7043


In [7]:

# Explore the Offer column
print("Offer Distribution:")
print(df["Offer"].value_counts(dropna=False))

# Compare customers without an accepted offer by Customer Status
print("\nOffer = None by Customer Status:")

offer_analysis = pd.crosstab(
    df["Customer Status"],
    df["Offer"].eq("None"),
    rownames=["Customer Status"],
    colnames=["No Accepted Offer"],
    margins=True
)

display(offer_analysis)


Offer Distribution:
Offer
None       3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

Offer = None by Customer Status:


No Accepted Offer,False,True,All
Customer Status,,,
Churned,818,1051,1869
Joined,175,279,454
Stayed,2173,2547,4720
All,3166,3877,7043


In [8]:

# Reload dataset preserving "None" as a valid category
df = pd.read_csv(
    file_path,
    keep_default_na=False,
    na_values=[""]
)


# Verify Offer distribution
print("Offer Distribution:")
print(df["Offer"].value_counts(dropna=False))

# Check actual missing values
print("\nMissing Values in Offer:")
print(df["Offer"].isna().sum())

# Count customers without an accepted offer
print("\nCustomers without an accepted offer:")
print((df["Offer"] == "None").sum())

# Verify remaining columns with missing values
print("\nColumns with missing values:")
print((df.isna().sum() > 0).sum())


Offer Distribution:
Offer
None       3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

Missing Values in Offer:
0

Customers without an accepted offer:
3877

Columns with missing values:
14


In [9]:

# Analyze Customer Status distribution
print("Customer Status Distribution:")
print(df["Customer Status"].value_counts(dropna=False))

# Calculate percentages
print("\nCustomer Status Percentage:")
print(
    df["Customer Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Validate Churn Reason availability
print("\nChurn Reason by Customer Status:")

churn_validation = pd.crosstab(
    df["Customer Status"],
    df["Churn Reason"].notna(),
    rownames=["Customer Status"],
    colnames=["Churn Reason Available"],
    margins=True
)

display(churn_validation)


Customer Status Distribution:
Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

Customer Status Percentage:
Customer Status
Stayed     67.02
Churned    26.54
Joined      6.45
Name: proportion, dtype: float64

Churn Reason by Customer Status:


Churn Reason Available,False,True,All
Customer Status,,,
Churned,0,1869,1869
Joined,454,0,454
Stayed,4720,0,4720
All,5174,1869,7043



Data Quality Analysis & Key Findings

1. Dataset Overview
- Total customers: 7,043
- Total features and target-related columns: 38
- Dataset source: Maven Analytics - Telecom Customer Churn

2. Internet Service - Structural Missingness
- 1,526 customers do not have internet service
- Ten internet-related columns contain missing values for exactly these customers
- No missing values were found in these columns among customers with internet service

Finding: Missing values are structurally related to internet service availability

3. Phone Service - Structural Missingness
- 682 customers do not have phone service.
- Multiple Lines and Avg Monthly Long Distance Charges are missing for exactly these customers.

Finding: Missing values are structurally related to phone service availability.

4. Offer - Valid Category
- 3,877 customers have Offer = None
- The original CSV uses "None" as a valid category
- Pandas initially interpreted this category as a missing value

Resolution: The CSV was reloaded using keep_default_na=False and na_values=[""]

5. Customer Status
- Stayed: 4,720 customers
- Churned: 1,869 customers
- Joined: 454 customers

Churn Reason is available exclusively for churned customers

ML Consideration: Churn Reason and Churn Category must be excluded from model features to prevent target leakage

6. Data Preparation Strategy
- Preserve the original CSV files
- Avoid deleting customers with structural missing values
- Handle missing values according to business meaning
- Investigate potential target leakage before model training
- Apply preprocessing transformations within the ML pipeline


In [13]:

# Display all dataset columns
print("Dataset Columns:")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

print("\nTotal columns:", len(df.columns))


Dataset Columns:
1. Customer ID
2. Gender
3. Age
4. Married
5. Number of Dependents
6. City
7. Zip Code
8. Latitude
9. Longitude
10. Number of Referrals
11. Tenure in Months
12. Offer
13. Phone Service
14. Avg Monthly Long Distance Charges
15. Multiple Lines
16. Internet Service
17. Internet Type
18. Avg Monthly GB Download
19. Online Security
20. Online Backup
21. Device Protection Plan
22. Premium Tech Support
23. Streaming TV
24. Streaming Movies
25. Streaming Music
26. Unlimited Data
27. Contract
28. Paperless Billing
29. Payment Method
30. Monthly Charge
31. Total Charges
32. Total Refunds
33. Total Extra Data Charges
34. Total Long Distance Charges
35. Total Revenue
36. Customer Status
37. Churn Category
38. Churn Reason

Total columns: 38


In [14]:

# Define columns by their role

target_columns = [
    "Customer Status"
]

identifier_columns = [
    "Customer ID"
]

leakage_columns = [
    "Churn Category",
    "Churn Reason"
]

temporal_review_columns = [
    "Number of Referrals",
    "Tenure in Months",
    "Avg Monthly Long Distance Charges",
    "Avg Monthly GB Download",
    "Total Charges",
    "Total Refunds",
    "Total Extra Data Charges",
    "Total Long Distance Charges",
    "Total Revenue"
]

# Create a feature audit table
feature_audit = pd.DataFrame({
    "Column": df.columns
})

feature_audit["Role"] = "Candidate Feature - Pending Review"

# Assign roles
feature_audit.loc[
    feature_audit["Column"].isin(target_columns),
    "Role"
] = "Target"

feature_audit.loc[
    feature_audit["Column"].isin(identifier_columns),
    "Role"
] = "Identifier"

feature_audit.loc[
    feature_audit["Column"].isin(leakage_columns),
    "Role"
] = "Direct Leakage"

feature_audit.loc[
    feature_audit["Column"].isin(temporal_review_columns),
    "Role"
] = "Temporal Review"

# Display the complete audit
display(feature_audit)

# Summary
print("\nFeature Audit Summary:")
print(feature_audit["Role"].value_counts())


,Column,Role
0,Customer ID,Identifier
1,Gender,Candidate Feature - Pending Review
2,Age,Candidate Feature - Pending Review
3,Married,Candidate Feature - Pending Review
4,Number of Dependents,Candidate Feature - Pending Review
5,City,Candidate Feature - Pending Review
6,Zip Code,Candidate Feature - Pending Review
7,Latitude,Candidate Feature - Pending Review
8,Longitude,Candidate Feature - Pending Review
9,Number of Referrals,Temporal Review



Feature Audit Summary:
Role
Candidate Feature - Pending Review    25
Temporal Review                        9
Direct Leakage                         2
Identifier                             1
Target                                 1
Name: count, dtype: int64


In [15]:

# Locate the data dictionary
dictionary_path = (
    project_root
    / "data"
    / "raw"
    / "telecom_data_dictionary.csv"
)

# Load the data dictionary
try:
    data_dictionary = pd.read_csv(
        dictionary_path,
        encoding="utf-8-sig",
        keep_default_na=False
    )
except UnicodeDecodeError:
    data_dictionary = pd.read_csv(
        dictionary_path,
        encoding="cp1252",
        keep_default_na=False
    )

# Display dictionary information
print("Data Dictionary Overview:")
print("Total rows:", data_dictionary.shape[0])
print("Total columns:", data_dictionary.shape[1])

print("\nDictionary column names:")
print(data_dictionary.columns.tolist())

display(data_dictionary.head(12))


Data Dictionary Overview:
Total rows: 40
Total columns: 3

Dictionary column names:
['Table', 'Field', 'Description']


,Table,Field,Description
0,Customer Churn,CustomerID,A unique ID that identifies each customer
1,Customer Churn,Gender,"The customer’s gender: Male, Female"
2,Customer Churn,Age,"The customer’s current age, in years, at the t..."
3,Customer Churn,Married,"Indicates if the customer is married: Yes, No"
4,Customer Churn,Number of Dependents,Indicates the number of dependents that live w...
5,Customer Churn,City,The city of the customer’s primary residence i...
6,Customer Churn,Zip Code,The zip code of the customer’s primary residence
7,Customer Churn,Latitude,The latitude of the customer’s primary residence
8,Customer Churn,Longitude,The longitude of the customer’s primary residence
9,Customer Churn,Number of Referrals,Indicates the number of times the customer has...


In [16]:

# Select variables for temporal and leakage review
fields_to_review = [
    "Tenure in Months",
    "Total Charges",
    "Total Revenue",
    "Number of Referrals",
    "Offer",
    "Contract",
    "Customer Status",
    "Internet Type",
    "Avg Monthly GB Download"
]

# Filter the original data dictionary
selected_definitions = data_dictionary[
    data_dictionary["Field"].isin(fields_to_review)
]

# Print full descriptions without truncation
for _, row in selected_definitions.iterrows():
    print(f"\nFIELD: {row['Field']}")
    print(f"DESCRIPTION: {row['Description']}")
    print("-" * 65)



FIELD: Number of Referrals
DESCRIPTION: Indicates the number of times the customer has referred a friend or family member to this company to date
-----------------------------------------------------------------

FIELD: Tenure in Months
DESCRIPTION: Indicates the total amount of months that the customer has been with the company by the end of the quarter specified above
-----------------------------------------------------------------

FIELD: Offer
DESCRIPTION: Identifies the last marketing offer that the customer accepted: None, Offer A, Offer B, Offer C, Offer D, Offer E
-----------------------------------------------------------------

FIELD: Internet Type
DESCRIPTION: Indicates the customer's type of internet connection: DSL, Fiber Optic, Cable (if the customer is not subscribed to internet service, this will be None)
-----------------------------------------------------------------

FIELD: Avg Monthly GB Download
DESCRIPTION: Indicates the customer’s average download volume in gi

In [17]:

# Select Customer Churn dictionary fields
churn_dictionary = data_dictionary.loc[
    data_dictionary["Table"] == "Customer Churn",
    ["Field", "Description"]
].copy()

# Normalize the Customer ID column name
churn_dictionary["Field"] = churn_dictionary["Field"].replace({
    "CustomerID": "Customer ID"
})

# Join Feature Audit with Data Dictionary
feature_audit_enriched = feature_audit.merge(
    churn_dictionary,
    left_on="Column",
    right_on="Field",
    how="left",
    validate="one_to_one"
)

# Remove redundant column
feature_audit_enriched = feature_audit_enriched.drop(
    columns=["Field"]
)

# Verify results
print("Total audited columns:", len(feature_audit_enriched))

print(
    "Missing descriptions:",
    feature_audit_enriched["Description"].isna().sum()
)

display(feature_audit_enriched)


Total audited columns: 38
Missing descriptions: 0


,Column,Role,Description
0,Customer ID,Identifier,A unique ID that identifies each customer
1,Gender,Candidate Feature - Pending Review,"The customer’s gender: Male, Female"
2,Age,Candidate Feature - Pending Review,"The customer’s current age, in years, at the t..."
3,Married,Candidate Feature - Pending Review,"Indicates if the customer is married: Yes, No"
4,Number of Dependents,Candidate Feature - Pending Review,Indicates the number of dependents that live w...
5,City,Candidate Feature - Pending Review,The city of the customer’s primary residence i...
6,Zip Code,Candidate Feature - Pending Review,The zip code of the customer’s primary residence
7,Latitude,Candidate Feature - Pending Review,The latitude of the customer’s primary residence
8,Longitude,Candidate Feature - Pending Review,The longitude of the customer’s primary residence
9,Number of Referrals,Temporal Review,Indicates the number of times the customer has...


In [18]:
print("Total columns:", len(feature_audit_enriched))

print(
    "Missing descriptions:",
    feature_audit_enriched["Description"].isna().sum()
)

print(
    "Empty descriptions:",
    feature_audit_enriched["Description"]
    .astype(str).str.strip().eq("").sum()
)

Total columns: 38
Missing descriptions: 0
Empty descriptions: 0
